# Clean test on a free Google computer: score the reports with Llama 3.2

This notebook scores the same 525 earnings reports with **Llama 3.2 (3 billion parameters)**, a model whose
training data ends in December 2023, on a free Google Colab machine with a graphics card. It takes about
30 to 40 minutes and uses no Claude account.

**Before you start:** in the menu choose **Runtime → Change runtime type → T4 GPU → Save**.
To run the larger Llama 3.1 8B check instead, change `CONFIG` in step 0 first.
Then choose **Runtime → Run all**.

* Step 3 asks for permission to use your Google Drive. Every answer is saved there as it arrives, so if the
  session disconnects, nothing is lost: just open the notebook again and **Run all**; it continues where it stopped.
* The first time, step 4 asks you to choose `colab_upload.zip` from your project folder (it is kept in your
  Drive, so later runs skip the upload).
* Keep this tab in front now and then: free sessions can disconnect when a tab is left idle in the background.
* The last cell downloads `colab_results.zip`.

## 0. Choose the model


In [ ]:
# Which model to run. Llama 3.2 3B is the default clean test; Llama 3.1 8B is the larger
# optional check (same December 2023 cutoff, about twice as slow).
CONFIG = "config/llama.yaml"        # or "config/llama8b.yaml"
MODEL = {"config/llama.yaml": "llama3.2:3b", "config/llama8b.yaml": "llama3.1:8b"}[CONFIG]
print("Running", MODEL, "with", CONFIG)

## 1. Check that a graphics card is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No graphics card: choose Runtime > Change runtime type > T4 GPU, then run this cell again"

## 2. Install Ollama, start it, and download Llama 3.2 (about 2 minutes)

In [ ]:
!sudo apt-get -qq install -y pciutils lshw zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
import os, shutil, subprocess, time, requests
assert shutil.which("ollama"), "Ollama did not install: see the messages above"
env = dict(os.environ, OLLAMA_NUM_PARALLEL="2")  # score two reports at a time on the graphics card
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}

## 3. Download the project code and connect your Google Drive

In [ ]:
%cd /content
!rm -rf llm-earnings-signal
!git clone -q https://github.com/pooja003-cloud/llm-earnings-signal.git
%cd /content/llm-earnings-signal
!pip install -q -e . 2>&1 | tail -1
!grep -q parse_json_answer src/earnsig/llm_score.py && echo "Project ready" || echo "Old project version: push the latest version to GitHub first"
!sed -i 's/^  workers: 1 /  workers: 2 /' config/llama.yaml  # applies to both configs (8B inherits it)
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/earnsig_" + CONFIG.split("/")[-1].removesuffix(".yaml")
os.makedirs(SAVE, exist_ok=True)
print("Answers will be saved in Google Drive:", SAVE)

## 4. Load the reports

Choose `colab_upload.zip` from your project folder. After a disconnect you can click **Cancel upload** instead to reuse the copy saved in your Drive.

In [ ]:
from google.colab import files
zip_in_drive = os.path.join(SAVE, "colab_upload.zip")
if not os.path.exists("data/events.csv"):
    print("Choose colab_upload.zip from your project folder,")
    print("or click Cancel upload to reuse the copy saved in your Google Drive.")
    uploaded = files.upload()
    if uploaded:
        shutil.copy(next(iter(uploaded)), zip_in_drive)
assert os.path.exists(zip_in_drive), "No colab_upload.zip yet: run this cell again and choose the file"
!unzip -oq "{zip_in_drive}"

def lines(p):
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0

saved, local = os.path.join(SAVE, "llm_cache.jsonl"), "data/llm_cache.jsonl"
if lines(saved) < lines(local):
    shutil.copy(local, saved)   # first run: start the Drive copy from your Mac's answers
os.remove(local)
os.symlink(saved, local)        # every new answer is written straight to Drive
print("Reports:", len(os.listdir("data/raw/docs")), "companies;", lines(saved), "saved answers so far")

## 5. Score the reports (about 30 minutes)

It prints progress every 50 reports. If the session disconnects, open the notebook again and **Run all**.

In [ ]:
!earnsig --config {CONFIG} score

## 6. Score 40 reports a second time to check consistency (about 2 minutes)

In [ ]:
!earnsig --config {CONFIG} consistency

## 7. Download the results

Put `colab_results.zip` in your project folder on your Mac.

In [ ]:
import sys; sys.path.insert(0, 'src')  # the project code, without restarting the session
from earnsig.config import load_config
cfg = load_config(CONFIG)
scores = cfg["paths"].llm_scores.relative_to(cfg["paths"].data.parent)
consist = (cfg["paths"].results / "consistency.json").relative_to(cfg["paths"].data.parent)
!zip -q colab_results.zip data/llm_cache.jsonl {scores} {consist} && unzip -l colab_results.zip
shutil.copy("colab_results.zip", SAVE)  # a copy also stays in your Google Drive
files.download("colab_results.zip")